# ML Zoomcamp 2026 — Homework 4 (Evaluation Metrics for Classification)
**Nicolau Calado Jofilsan** · 2026 cohort (DataTalks.Club)

**EN.** Same lead-scoring task as homework 3 (predict `converted`), now scored with ROC AUC,
precision, recall, F1 and cross-validation instead of plain accuracy. Dataset: the pinned 2026
release `course_lead_scoring_2026.csv` from the course repository (`cohorts/2026/data/`).
Deadline: Mon 19/10/2026, 20:00 (account timezone).

**PT.** Mesma tarefa de leads da atividade 3 (prever `converted`), agora avaliada por AUC da ROC,
precisão, revocação, F1 e validação cruzada, em vez de só acurácia. Base: versão fixada de 2026,
`course_lead_scoring_2026.csv`, do repositório do curso (`cohorts/2026/data/`).
Prazo: seg 19/10/2026, 20:00 (horário da conta).

Topics / Conteúdo: ROC AUC as a feature-ranking score, model AUC, precision-recall threshold
curves, F1, K-fold cross-validation, regularization (`C`) selected by CV.

## 0. Setup and data / Preparação e dados

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 50)

DATA_FILE = "course_lead_scoring_2026.csv"
DATA_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/data/course_lead_scoring_2026.csv"
)

if not os.path.exists(DATA_FILE):
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

df = pd.read_csv(DATA_FILE)
print("shape:", df.shape)
df.head()

shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## 1. Data preparation / Preparação dos dados

**EN.** Same rule as the homework asks: `'NA'` for missing categorical values, `0.0` for missing
numerical ones. Note the split here uses `random_state=1`, not 42 as in homework 3 — this
assignment defines its own split from scratch.
**PT.** Mesma regra do enunciado: `'NA'` nas categóricas e `0.0` nas numéricas. O split aqui usa
`random_state=1`, não 42 como na atividade 3 — este enunciado define sua própria divisão.

In [2]:
print(df.isnull().sum())

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]
target = "converted"

for c in categorical:
    df[c] = df[c].fillna("NA")

for c in numerical:
    df[c] = df[c].fillna(0.0)

print("\nmissing values after filling:", df.isnull().sum().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

missing values after filling: 0


## Split the data / Divisão dos dados

**EN.** 60/20/20, exactly as specified: first 80/20, then 75/25 of the first part, both with
`random_state=1`.
**PT.** 60/20/20, exatamente como no enunciado: primeiro 80/20, depois 75/25 da primeira parte,
ambos com `random_state=1`.

In [3]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values
y_full_train = df_full_train[target].values

# the target must not stay among the features / o alvo não pode ficar entre as variáveis
X_train_df = df_train.drop(columns=[target])
X_val_df = df_val.drop(columns=[target])
X_test_df = df_test.drop(columns=[target])
X_full_train_df = df_full_train.drop(columns=[target])

features = categorical + numerical

print("train:", len(df_train), "| val:", len(df_val), "| test:", len(df_test))
print("conversion rate in train / taxa de conversão no treino:", round(y_train.mean(), 4))

train: 3000 | val: 1000 | test: 1000
conversion rate in train / taxa de conversão no treino: 0.5763


## Q1. ROC AUC feature importance / Importância das variáveis pela AUC

**EN.** Use each numerical variable, on its own, as a score for `converted` and compute its AUC
against the training labels — no model yet, just the raw value. A variable with AUC below 0.5 is
negated first (it is still informative, just pointing the wrong way): negating it is the same as
using `1 - AUC` as its score.
**PT.** Usar cada variável numérica, sozinha, como pontuação para `converted` e calcular a AUC
contra os rótulos de treino — sem modelo ainda, só o valor bruto. Uma variável com AUC abaixo de
0,5 é invertida primeiro (ainda é informativa, só aponta na direção errada): inverter equivale a
usar `1 - AUC` como pontuação.

In [4]:
auc_scores = {}
for col in numerical:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    auc_scores[col] = auc
    print(f"{col:>26}: {auc:.4f}")

print("\nQ1 =", max(auc_scores, key=auc_scores.get))

             annual_income: 0.6082
  number_of_courses_viewed: 0.7230
         interaction_count: 0.7649
                lead_score: 0.7882

Q1 = lead_score


**Answer / Resposta Q1: `lead_score`** — AUC 0.7882, ahead of `interaction_count` (0.7649),
`number_of_courses_viewed` (0.7230) and `annual_income` (0.6082, the only one below 0.5 before
inverting — the raw sign of `annual_income` points slightly the wrong way).

EN: this ranks each variable by how well it alone separates converted from non-converted leads,
which is a quick screen before building any model.
PT: isso ordena cada variável por quão bem ela sozinha separa leads convertidos de não convertidos,
uma checagem rápida antes de montar qualquer modelo.

## Q2. Training the model / Treinamento do modelo

**EN.** One-hot encoding with `DictVectorizer`, fitted on the training set only and reused on
validation. Logistic regression with `solver='liblinear'`, `C=1.0`, `max_iter=1000`, as specified.
**PT.** One-hot encoding com `DictVectorizer`, ajustado só no treino e reaplicado na validação.
Regressão logística com `solver='liblinear'`, `C=1.0`, `max_iter=1000`, como no enunciado.

In [5]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(X_train_df[features].to_dict(orient="records"))
X_val = dv.transform(X_val_df[features].to_dict(orient="records"))

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_val)[:, 1]
auc_val = roc_auc_score(y_val, y_pred_proba)
print("validation AUC:", auc_val)
print("Q2 =", round(auc_val, 3))

validation AUC: 0.7318222288173319
Q2 = 0.732


**Answer / Resposta Q2: 0.732** — 0.7318 before rounding.

EN: a model built from all eight features (four categorical, four numerical) already beats every
single-variable score from Q1, including `lead_score` alone (0.788 on train vs. this 0.732 on an
unseen validation split) — a reminder that AUC is not directly comparable across different data
splits, only within the same one.
PT: um modelo com as oito variáveis (quatro categóricas, quatro numéricas) já supera qualquer
pontuação isolada da Q1, inclusive o `lead_score` sozinho (0,788 no treino contra este 0,732 na
validação, um split diferente) — lembrete de que a AUC só é diretamente comparável dentro do mesmo
split.

## Q3. Precision and recall / Precisão e revocação

**EN.** Scan thresholds from 0.00 to 1.00 in steps of 0.01 (101 values). At each threshold, turn
the validation probabilities into 0/1 predictions and compute precision and recall. Thresholds
where both are zero (no positive prediction at all) are ignored. The answer is the threshold
closest to where the two curves cross.
**PT.** Varrer limiares de 0,00 a 1,00 em passos de 0,01 (101 valores). Em cada limiar, transformar
as probabilidades de validação em previsões 0/1 e calcular precisão e revocação. Limiares em que as
duas são zero (nenhuma previsão positiva) são ignorados. A resposta é o limiar mais próximo de onde
as duas curvas se cruzam.

In [6]:
thresholds = np.arange(0.0, 1.01, 0.01)
precisions, recalls, f1_scores = [], [], []

for t in thresholds:
    y_pred = (y_pred_proba >= t).astype(int)
    tp = ((y_pred == 1) & (y_val == 1)).sum()
    fp = ((y_pred == 1) & (y_val == 0)).sum()
    fn = ((y_pred == 0) & (y_val == 1)).sum()

    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f1)

precisions = np.array(precisions)
recalls = np.array(recalls)
f1_scores = np.array(f1_scores)

valid = ~((precisions == 0) & (recalls == 0))
diffs = np.abs(precisions - recalls)
diffs_for_search = np.where(valid, diffs, np.inf)

idx_q3 = int(np.argmin(diffs_for_search))
print("around the crossing point / ao redor do cruzamento:")
for i in range(max(0, idx_q3 - 2), min(len(thresholds), idx_q3 + 3)):
    print(f"  t={thresholds[i]:.2f}  precision={precisions[i]:.4f}  recall={recalls[i]:.4f}  "
          f"|diff|={diffs[i]:.4f}")

print("\nQ3 =", round(thresholds[idx_q3], 2))

around the crossing point / ao redor do cruzamento:
  t=0.61  precision=0.7190  recall=0.7730  |diff|=0.0540
  t=0.62  precision=0.7259  recall=0.7457  |diff|=0.0198
  t=0.63  precision=0.7328  recall=0.7253  |diff|=0.0075
  t=0.64  precision=0.7299  recall=0.6962  |diff|=0.0336
  t=0.65  precision=0.7410  recall=0.6689  |diff|=0.0721

Q3 = 0.63


**Answer / Resposta Q3: 0.63** — precision 0.7328, recall 0.7253, a difference of 0.0075, the
smallest in the whole scan.

EN: below 0.63 the model favors recall (it flags more leads, catching more of the real converters
at the cost of more false alarms); above 0.63 it favors precision. This is the point where neither
error type is being traded off against the other.
PT: abaixo de 0,63 o modelo favorece a revocação (sinaliza mais leads, pegando mais conversões
reais ao custo de mais alarmes falsos); acima de 0,63 favorece a precisão. É o ponto em que nenhum
dos dois erros está sendo privilegiado.

## Q4. F1 score / Pontuação F1

**EN.** Using the same threshold scan, compute F1 = 2·P·R / (P + R) at every point and find where
it peaks.
**PT.** Usando a mesma varredura de limiares, calcular F1 = 2·P·R / (P + R) em cada ponto e
encontrar o pico.

In [7]:
idx_q4 = int(np.argmax(f1_scores))
for i in range(max(0, idx_q4 - 2), min(len(thresholds), idx_q4 + 3)):
    print(f"  t={thresholds[i]:.2f}  F1={f1_scores[i]:.4f}")

print("\nQ4 =", round(thresholds[idx_q4], 2))

  t=0.39  F1=0.7524
  t=0.40  F1=0.7535
  t=0.41  F1=0.7576
  t=0.42  F1=0.7563
  t=0.43  F1=0.7560

Q4 = 0.41


**Answer / Resposta Q4: 0.41** — F1 = 0.7576, the highest across the whole scan.

EN: the F1-optimal threshold (0.41) sits below the precision/recall crossing point (0.63) found in
Q3, because F1 favors whichever side of the trade-off still raises the harmonic mean, not the point
where the two curves meet.
PT: o limiar ótimo para F1 (0,41) fica abaixo do ponto de cruzamento de precisão/revocação (0,63) da
Q3, porque o F1 favorece o lado da troca que ainda eleva a média harmônica, não o ponto em que as
duas curvas se encontram.

## Q5. 5-fold cross-validation / Validação cruzada em 5 partes

**EN.** `KFold(n_splits=5, shuffle=True, random_state=1)` over `df_full_train` (train + validation
combined, 4000 rows). For each fold: fit `DictVectorizer` and the model on the 4 training folds,
score AUC on the held-out fold. Report the standard deviation across the 5 scores.
**PT.** `KFold(n_splits=5, shuffle=True, random_state=1)` sobre `df_full_train` (treino + validação
juntos, 4000 linhas). Em cada parte: ajustar `DictVectorizer` e o modelo nas 4 partes de treino,
pontuar a AUC na parte separada. Reportar o desvio-padrão entre as 5 pontuações.

In [8]:
def cv_auc_scores(C, feature_list):
    """5-fold AUC scores for a given C, re-fitting the encoder inside every fold."""
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    scores = []
    for train_idx, val_idx in kfold.split(X_full_train_df):
        df_t = X_full_train_df.iloc[train_idx]
        df_v = X_full_train_df.iloc[val_idx]
        y_t = y_full_train[train_idx]
        y_v = y_full_train[val_idx]

        dv_cv = DictVectorizer(sparse=False)
        X_t = dv_cv.fit_transform(df_t[feature_list].to_dict(orient="records"))
        X_v = dv_cv.transform(df_v[feature_list].to_dict(orient="records"))

        m = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
        m.fit(X_t, y_t)
        p = m.predict_proba(X_v)[:, 1]
        scores.append(roc_auc_score(y_v, p))
    return np.array(scores)


scores_q5 = cv_auc_scores(C=1.0, feature_list=features)
print("fold scores:", np.round(scores_q5, 4))
print("mean:", round(scores_q5.mean(), 4), "| std:", round(scores_q5.std(), 4))
print("\nQ5 =", round(scores_q5.std(), 3))

fold scores: [0.7306 0.7296 0.7363 0.7412 0.7209]
mean: 0.7317 | std: 0.0069

Q5 = 0.007


**Answer / Resposta Q5: 0.007** — mean AUC 0.7317, fold scores ranging from 0.7209 to 0.7412.

EN: a standard deviation this small, relative to the mean, says the model's performance is stable
across different 4000-row samples of the data — the 0.732 from Q2 was not a lucky split.
PT: um desvio-padrão tão pequeno frente à média diz que o desempenho do modelo é estável entre
amostras diferentes de 4000 linhas — o 0,732 da Q2 não foi sorte de split.

## Q6. Hyperparameter tuning / Ajuste de hiperparâmetro

**EN.** Same 5-fold setup, now comparing `C` in {0.000001, 0.001, 1}. Smaller `C` means stronger
regularization (coefficients pulled harder towards zero).
**PT.** Mesma configuração de 5 partes, agora comparando `C` em {0,000001; 0,001; 1}. `C` menor
significa regularização mais forte (coeficientes puxados com mais força para zero).

In [9]:
results_q6 = {}
for C in [0.000001, 0.001, 1]:
    s = cv_auc_scores(C=C, feature_list=features)
    results_q6[C] = (round(s.mean(), 3), round(s.std(), 3))
    print(f"C = {C:<10}: mean AUC = {s.mean():.4f}  std = {s.std():.4f}  "
          f"-> mean={round(s.mean(), 3)}  std={round(s.std(), 3)}")

best_c = max(results_q6, key=lambda c: (results_q6[c][0], -results_q6[c][1], -c))
print("\nQ6 =", best_c)

C = 1e-06     : mean AUC = 0.6167  std = 0.0187  -> mean=0.617  std=0.019


C = 0.001     : mean AUC = 0.7492  std = 0.0100  -> mean=0.749  std=0.01


C = 1         : mean AUC = 0.7317  std = 0.0069  -> mean=0.732  std=0.007

Q6 = 0.001


**Answer / Resposta Q6: 0.001** — mean AUC 0.749 (std 0.010), ahead of `C=1` (0.732, std 0.007)
and far ahead of `C=0.000001` (0.617, std 0.019).

EN: this is worth pausing on — mild regularization (`C=0.001`) generalizes *better* on average
than no regularization at all (`C=1`, the value used in Q2 and Q5). The extreme case,
`C=0.000001`, over-shrinks the coefficients and collapses toward a near-constant score, which both
lowers the mean AUC and makes it less stable across folds (std 0.019, the highest of the three).
PT: vale parar nesse ponto — uma regularização branda (`C=0,001`) generaliza *melhor*, em média, do
que nenhuma regularização (`C=1`, usado na Q2 e na Q5). O caso extremo, `C=0,000001`, encolhe demais
os coeficientes e colapsa para uma pontuação quase constante, o que reduz a AUC média e também a
torna menos estável entre as partes (desvio-padrão 0,019, o maior dos três).

### Homework 4 — answers / respostas

| Question | Answer |
|---|---|
| Q1 Highest single-variable AUC | **`lead_score`** (0.7882) |
| Q2 Validation AUC | **0.732** |
| Q3 Precision/recall crossing threshold | **0.63** |
| Q4 Best F1 threshold | **0.41** |
| Q5 5-fold AUC std (`C=1`) | **0.007** |
| Q6 Best `C` | **0.001** (mean AUC 0.749) |

### Notes / Observações

**EN.** 1) `DictVectorizer` is re-fitted inside every CV fold, not once on the full training set,
so no fold ever sees an encoding built from data outside it. 2) The train/val/test split here uses
`random_state=1`, as this homework specifies — different from homework 3's `random_state=42` — so
the two activities are not numerically comparable row-for-row even though they share a dataset.
3) Q6 is a case where trusting the metric over the default hyperparameter mattered: `C=1` is
scikit-learn's default, but `C=0.001` scored higher on every fold-based check.

**PT.** 1) O `DictVectorizer` é reajustado dentro de cada parte da validação cruzada, não uma única
vez no treino completo, para que nenhuma parte veja uma codificação construída com dados de fora
dela. 2) O split treino/validação/teste aqui usa `random_state=1`, como pede este enunciado —
diferente do `random_state=42` da atividade 3 — então as duas atividades não são comparáveis
número a número, mesmo usando a mesma base. 3) A Q6 é um caso em que confiar na métrica, e não no
hiperparâmetro padrão, fez diferença: `C=1` é o padrão do scikit-learn, mas `C=0,001` pontuou mais
alto em toda checagem por partes.